# Week 5: Semantic Search

Goal: find listings by meaning instead of exact words, and compare it with keyword search. Only the first few words of each remark are shown so the notebook is safe to commit.

In [1]:
import logging
import re
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import faiss
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from scripts.query_parser import QueryParser
from scripts.semantic_search import KeywordSearcher, SemanticSearcher, load_listings

pd.set_option("display.max_colwidth", 100)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

## 1. The search index

Each listing's cleaned remarks are turned into a vector of 384 numbers using the `all-MiniLM-L6-v2` model. Listings that mean similar things end up with similar vectors. FAISS stores the vectors so the closest ones to a query can be found quickly.

Keyword search (BM25) is the comparison. It ranks listings by how often the query words appear, giving more weight to rarer words.

In [2]:
listings = load_listings()
semantic = SemanticSearcher.load()

keyword = KeywordSearcher()
keyword.build_index(listings["remarks_clean"], listings["L_ListingID"])

print(f"Listings indexed: {semantic.index.ntotal:,}")
print(f"Numbers per listing: {semantic.index.d}")

Listings indexed: 54,838
Numbers per listing: 384


In [3]:
info = listings.set_index("L_ListingID")
ADDRESS = r"\b\d{2,6}\s+(?:[NSEW]\.?\s+)?(?:[A-Z][\w.']*\s+){1,3}(?:St|Street|Ave|Avenue|avenue|Rd|Road|Dr|Drive|Way|Ln|Lane|Blvd|boulevard|Ct|Court|Pl|Place|Cir)\b\.?"

def snippet(text):
    """First few words of the remarks, with street addresses hidden."""
    return re.sub(ADDRESS, "[address]", text)[:90] + "..."

def show(results):
    """Add city, price, and the first few words of the remarks to search results."""
    out = results.copy()
    out["city"] = out["listing_id"].map(info["L_City"])
    out["price"] = out["listing_id"].map(info["price"]).map("${:,.0f}".format)
    out["remarks start"] = out["listing_id"].map(info["remarks_clean"].map(snippet))
    return out

## 2. Example searches

In [4]:
show(semantic.search("quiet retreat surrounded by nature", top_k=5))

,listing_id,score,city,price,remarks start
0,1189490750,0.5752,Felton,"$715,000","Welcome to your private, single-level mountain retreat! Ideally located just off the main ..."
1,1175160202,0.5613,29 Palms,"$890,000",Escape to a secluded 20-acre corporate retreat compound designed for privacy and serenity....
2,1175527729,0.5443,Running Springs,"$249,999",Serene mountain retreat tucked among mature trees in Running Springs. This quaint home off...
3,1190153793,0.5289,El Sobrante,"$995,000","Welcome to your private retreat in Carriage Hills South, where space, nature, and convenie..."
4,1114253717,0.5231,Los Osos,"$3,175,000",A One-of-a-Kind Private Retreat Above the Pacific Tucked into the coastal hills of Los Oso...


## 3. Semantic vs keyword side by side

Same query, both methods. Keyword search needs the exact words. Semantic search matches the idea.

In [5]:
query = "great for entertaining with a pool and big backyard"
pd.concat({"semantic": show(semantic.search(query, top_k=5))[["listing_id", "city", "remarks start"]],
           "keyword": show(keyword.search(query, top_k=5))[["listing_id", "city", "remarks start"]]}, axis=1)

semantic            \
   listing_id      city   
0  1190645439     Vista   
1  1183227015   Ontario   
2  1190952827     Indio   
3  1158624641     Chico   
4  1157874412  Highland   

                                                                                                  \
                                                                                   remarks start   
0  This completely remodeled and thoughtfully reimagined luxury residence offers the rare com...   
1  Welcome to this beautifully upgraded single-story pool home, tucked away in a quiet cul-de...   
2  Professionally Designed Resort-Style Backyard | Nearly Half-Acre | 4 En Suite Bedrooms | G...   
3  Welcome to a truly one-of-a-kind resort compound in one of Chico's most sought-after neigh...   
4  4 bedroom pool home with spacious front yard and breathtaking views of SoCal mountains. In...   

      keyword               \
   listing_id         city   
0  1179292779      Brawley   
1  1177990582      Antioch   
2  1175344266  Palm Desert   
3  1188710543    Lancaster   
4  1178371695  Lake Forest   

                                                                                                  
                                                                                   remarks start  
0  This is a great opportunity to purchase an affordable home in a great location in Brawley ...  
1  Vacation at home in this stunning 4 bedroom, 3 bath, Former Model home with a tropical fee...  
2  price improvementBeautiful Newer Home in Palm Desert! Completed in 2025 This great propert...  
3  Great opportunity in a good neighborhood of Palmdale. This Home is located on a quiet priv...  
4  Upgraded single level home with big front yard and big back yard. New flooring, window shu...

## 4. Relevance evaluation

10 queries, top 5 results from each method = 100 query-result pairs (50 per method).

To judge fairly, I pooled each query's results from both methods, shuffled them, and marked each one relevant or not **without knowing which method found it**. What counts as relevant for each query is written down in `data/processed/search_eval_queries.csv`. The judgments are in `data/processed/search_relevance.csv`.

The score is **precision@5** = the share of the top 5 results that are relevant.

In [6]:
queries = pd.read_csv(ROOT / "data/processed/search_eval_queries.csv").set_index("query_id")
judged = pd.read_csv(ROOT / "data/processed/search_relevance.csv")

by_query = judged.pivot_table(index="query_id", columns="method", values="relevant", aggfunc="mean")
by_query.insert(0, "query", queries["query"])
by_query.insert(1, "style", queries["style"])
by_query.loc["overall"] = ["", "", judged[judged.method == "keyword"].relevant.mean(),
                           judged[judged.method == "semantic"].relevant.mean()]
by_query[["keyword", "semantic"]] = (by_query[["keyword", "semantic"]].astype(float) * 100).round().astype(int).astype(str) + "%"
by_query

method,query,style,keyword,semantic
query_id,,,,
S01,quiet retreat surrounded by nature,descriptive,100%,100%
S02,room for multigenerational living,descriptive,100%,100%
S03,fixer upper with lots of potential,descriptive,100%,100%
S04,ocean view condo near the beach,descriptive,100%,100%
S05,dream kitchen for someone who loves to cook,descriptive,80%,100%
S06,great for entertaining with a pool and big backyard,descriptive,20%,100%
S07,space to work from home,descriptive,80%,60%
S08,horse property with acreage,keyword,80%,100%
S09,RV parking,keyword,100%,100%


In [7]:
shared = sum(len(set(g.loc[g.method == "semantic", "listing_id"]) & set(g.loc[g.method == "keyword", "listing_id"]))
             for _, g in judged.groupby("query_id"))
print(f"Results found by both methods: {shared} of 50")

Results found by both methods: 3 of 50


### Where each method went wrong

| Query | Method | What happened |
|---|---|---|
| great for entertaining with a pool and big backyard | keyword | Matched "big backyard **for a** pool", homes with room to add a pool but no pool yet |
| energy efficient home with paid off solar | semantic | Returned homes with **leased** solar. "Leased" and "paid off" solar read as similar meaning to the model |
| space to work from home | both | Returned homes that mention "work space" in the kitchen or have no office at all |
| horse property with acreage | keyword | One listing only had "room for a horse ranch", not an actual horse setup |

## 5. Speed

Target: under 100 ms per search with 10,000 listings. Times include turning the query into a vector.

In [8]:
def median_ms(search, runs=30):
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        search()
        times.append((time.perf_counter() - start) * 1000)
    return np.median(times)

query = "quiet retreat surrounded by nature"
query_vector = semantic.embed([query])

# 10,000-listing index made from the first 10,000 vectors
small = faiss.IndexFlatIP(semantic.index.d)
small.add(semantic.index.reconstruct_n(0, 10_000))

speed = pd.DataFrame({
    "listings": ["10,000", f"{semantic.index.ntotal:,}", f"{semantic.index.ntotal:,}", f"{semantic.index.ntotal:,}"],
    "what is timed": ["semantic (embed query + search)", "semantic (embed query + search)",
                      "semantic (search only)", "keyword (BM25)"],
    "median ms": [median_ms(lambda: small.search(semantic.embed([query]), 10)),
                  median_ms(lambda: semantic.search(query, top_k=10)),
                  median_ms(lambda: semantic.index.search(query_vector, 10)),
                  median_ms(lambda: keyword.search(query, top_k=10), runs=5)],
})
speed["median ms"] = speed["median ms"].round(1)
speed

,listings,what is timed,median ms
0,"10,000",semantic (embed query + search),7.7
1,"54,838",semantic (embed query + search),10.3
2,"54,838",semantic (search only),3.1
3,"54,838",keyword (BM25),75.7


## 6. Combining with Week 4 filters

The Week 4 parser handles hard filters (bedrooms, price, city). Semantic search can then rank only the listings that pass. Here the filters are applied with pandas so this runs without MySQL.

In [9]:
filters = QueryParser().parse("3 bed under 900k in San Diego")
print("Filters:", filters)

passes = listings[(listings["beds"] == filters["bedrooms"])
                  & (listings["price"] <= filters["price_max"])
                  & (listings["L_City"].isin(filters["city"]))]
print(f"Listings that pass the filters: {len(passes)}")

show(semantic.search("quiet street with a big backyard", top_k=5, only_ids=passes["L_ListingID"]))

Filters: {'bedrooms': 3, 'price_max': 900000, 'city': ['San Diego']}
Listings that pass the filters: 226


,listing_id,score,city,price,remarks start
0,1179206852,0.5013,San Diego,"$649,900",Great home with a lot of potential. Large Backyard with mature trees....
1,1190536139,0.5005,San Diego,"$774,999","Welcome to your new home at [address], a 3 bedroom, 2 bathroom home offering 1494 square f..."
2,1180694020,0.4788,San Diego,"$650,000",Welcome to this light and bright updated home with wood flooring/carpets. Wood shutter bli...
3,1177112519,0.4709,San Diego,"$869,000","Beautifully remodeled Mid-Century Modern home featuring 3 bedrooms, 2 bathrooms, a detache..."
4,1177912859,0.4639,San Diego,"$779,900","Welcome to this 3-bedroom, 2-bathroom home offering 1231 square feet of comfortable living..."


## 7. Key findings

**I. Results**

*Semantic search edges out keyword search, and both find mostly relevant listings.*
- Precision@5 of 90% for semantic vs 86% for keyword (50 results each)
- They tie on 6 of 10 queries, but rarely return the same listings (3 of 50 shared)

**II. Where each one is better**

*Semantic understands the idea, keyword is better at exact details.*
- Semantic wins descriptive searches like "great for entertaining with a pool and big backyard" (100% vs 20%)
- Keyword wins exact details like "paid off solar" (100% vs 40%), since the model treats leased and paid-off solar as similar

**III. Speed**

*Well under the 100 ms target.*
- Semantic search took 6 to 44 ms per search across runs, even for all 54,838 listings and including turning the query into a vector
- Keyword search took about 75 to 90 ms for all listings
- Times vary with what else the computer is doing, but stayed under 100 ms in every run

**IV. Next steps**
- Combine both methods (hybrid search) to get semantic's understanding plus keyword's precision on details
- Use Week 4 filters first, then rank the remaining listings with semantic search (Section 6)